# HingRoBERTa & XLM-RoBERTa — Training + Cross-Lingual Testing + XAI Outputs

## Pipeline
| Step | Detail |
|---|---|
| **Train** | `FINAL_NLP_DATASET.csv` — Hinglish (33,984 rows), no normalization |
| **Test 1** | `cleaned_final_english.csv` — English (24,783 rows) |
| **Test 2** | `cleaned_hindi_final.csv` — Hindi (8,192 rows) |
| **Models** | `l3cube-pune/hinglish-roberta` (HingRoBERTa) + `xlm-roberta-base` (XLM-RoBERTa) |
| **XAI** | Saves attentions, hidden states, logits, token rankings → ready for SHAP / Attention / AOPC |

**Runtime:** GPU → Runtime > Change runtime type > T4 GPU

---
### Label Convention (confirmed from data inspection)
- `0` = Non-Hate / Clean
- `1` = Hate / Offensive

All three datasets use this same convention.

## Cell 1 — Install Dependencies

In [1]:
%%capture
!pip install transformers==4.40.0 datasets scikit-learn accelerate emoji

## Cell 2 — Imports

In [2]:
# =========================================
# IMPORTS + SETUP
# =========================================
import os, gc, warnings, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import emoji as emoji_lib

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification , get_linear_schedule_with_warmup
from torch.optim import AdamW

# =========================================
# DEVICE + SEED
# =========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

print("Device:", device)


Device: cuda


## Cell 3 — Configuration
**Change only here — nothing is hardcoded elsewhere.**

In [3]:
# =========================================
# MODELS
# =========================================
MODELS = {
    'HingRoBERTa': 'l3cube-pune/hing-roberta',
    'XLM-RoBERTa': 'xlm-roberta-base',
}

# =========================================
# LABELS
# =========================================
NUM_LABELS  = 2
LABEL_NAMES = ['Non-Hate', 'Hate']   # 0, 1

# =========================================
# TRAINING HYPERPARAMETERS
# =========================================
EPOCHS         = 4          # 5 is okay, but 4 avoids overfitting on small data
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
MAX_SEQ_LEN    = 128

WARMUP_RATIO   = 0.1
WEIGHT_DECAY   = 0.01
GRAD_CLIP      = 1.0

VAL_SIZE       = 0.10       # IMPORTANT: match your earlier pipeline (10%)
RANDOM_SEED    = 42

# =========================================
# XAI SETTINGS
# =========================================
OUTPUT_ATTENTIONS    = True
OUTPUT_HIDDEN_STATES = True

XAI_SAMPLE_SIZE      = 200   # keep small to avoid memory crash
AOPC_STEPS           = 10

# =========================================
# OUTPUT PATHS
# =========================================
BASE_DIR = 'roberta_outputs'
MODEL_DIR = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
XAI_DIR = os.path.join(BASE_DIR, 'xai')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(XAI_DIR, exist_ok=True)

# =========================================
# SEED
# =========================================
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

print('✅ Config ready.')

✅ Config ready.


## Cell 4 — Emoji Cleaning
Removes low-signal positive emojis and caps repetitions at 2 per emoji.
No text normalization applied (as per project spec).

In [4]:
import collections
import re
import emoji as emoji_lib

def clean_emojis(text, max_per_emoji=2):
    """
    1. Remove ALL emojis
    2. (Structure kept same as original)
    3. Collapse extra whitespace
    """
    text = str(text)

    # STEP 1: Remove ALL emojis
    cleaned = ''.join('' if emoji_lib.is_emoji(ch) else ch for ch in text)

    # STEP 2: (kept for structure consistency, but no emojis remain)
    seen = collections.Counter()
    result = []
    for ch in cleaned:
        if emoji_lib.is_emoji(ch):  # will rarely trigger now
            seen[ch] += 1
            if seen[ch] <= max_per_emoji:
                result.append(ch)
        else:
            result.append(ch)

    # STEP 3: Clean whitespace
    return re.sub(r'\s+', ' ', ''.join(result)).strip()

## Cell 5 — Upload & Load Datasets

In [5]:
from google.colab import files
print('Upload 1/3 — FINAL_NLP_DATASET.csv (Hinglish TRAINING)')
u1 = files.upload()
hinglish_file = list(u1.keys())[0]

print('Upload 2/3 — cleaned_final_english.csv (English TEST)')
u2 = files.upload()
english_file = list(u2.keys())[0]

print('Upload 3/3 — cleaned_hindi_final.csv (Hindi TEST)')
u3 = files.upload()
hindi_file = list(u3.keys())[0]


Upload 1/3 — FINAL_NLP_DATASET.csv (Hinglish TRAINING)


Saving cleaned_hinglish.csv to cleaned_hinglish.csv
Upload 2/3 — cleaned_final_english.csv (English TEST)


Saving cleaned_english.csv to cleaned_english.csv
Upload 3/3 — cleaned_hindi_final.csv (Hindi TEST)


Saving cleaned_hindi.csv to cleaned_hindi.csv


In [6]:
def load_data(path, text_col, label_col, name):
    df = pd.read_csv(path)
    df = df[[text_col, label_col]].dropna().reset_index(drop=True)

    df['text']  = df[text_col].astype(str)
    df['label'] = df[label_col].astype(int)

    df = df[['text', 'label']]

    print(f"{name}: {len(df)} rows")

    return df

In [7]:
# =========================================
# LOAD DATASETS (ADD THIS HERE)
# =========================================
train_df = load_data(
    hinglish_file,
    'text',
    'label',
    'Hinglish'
)

english_df = load_data(
    english_file,
    'text',
    'label',
    'English'
)

hindi_df = load_data(
    hindi_file,
    'text',
    'label',
    'Hindi'
)

Hinglish: 33984 rows
English: 24781 rows
Hindi: 8192 rows


## Cell 6 — PyTorch Dataset & DataLoader

In [8]:
class HateSpeechDataset(Dataset):
    """Tokenized dataset. Keeps raw text alongside tensors for XAI reference."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            **{k: v.squeeze(0) for k, v in enc.items()},
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'raw_text': self.texts[idx],
        }


def xai_collate_fn(batch):
    """Handles the raw_text string field cleanly."""
    raw_texts = [b.pop('raw_text') for b in batch]
    labels    = torch.stack([b.pop('labels') for b in batch])
    keys      = list(batch[0].keys())
    tensors   = {k: torch.stack([b[k] for b in batch]) for k in keys}
    tensors['labels']    = labels
    tensors['raw_texts'] = raw_texts
    return tensors


def make_loader(texts, labels, tokenizer, batch_size, max_len, shuffle):
    ds = HateSpeechDataset(texts, labels, tokenizer, max_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True,
                      collate_fn=xai_collate_fn)

print('Dataset class ready.')

Dataset class ready.


## Cell 7 — Training & Evaluation Functions

In [9]:
def run_train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch.pop('raw_texts')
        labels = batch.pop('labels').to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        # During training: no attentions/hidden states (saves memory)
        out  = model(**inputs, output_attentions=False, output_hidden_states=False)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct    += (out.logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def run_eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            batch.pop('raw_texts')
            labels = batch.pop('labels').to(device)
            inputs = {k: v.to(device) for k, v in batch.items()}
            out    = model(**inputs, output_attentions=False, output_hidden_states=False)
            loss   = loss_fn(out.logits, labels)
            probs  = torch.softmax(out.logits, dim=-1)
            total_loss += loss.item()
            all_preds.extend(out.logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    accuracy = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), macro_f1, accuracy, all_preds, all_labels, all_probs

print('Training functions ready.')

Training functions ready.


## Cell 8 — Train Both Models on Hinglish
HingRoBERTa → XLM-RoBERTa sequentially.
Each model saves its best checkpoint (by Val Macro F1).

In [10]:
# =========================================
# STRATIFIED TRAIN / VALIDATION SPLIT
# =========================================
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=VAL_SIZE,          # use your config
    stratify=train_df['label'],
    random_state=RANDOM_SEED
)

print(f"Train: {len(X_tr):,} | Val: {len(X_val):,}")

# =========================================
# CLASS WEIGHTS (HANDLE IMBALANCE)
# =========================================
import numpy as np
import torch

class_counts = np.bincount(y_tr)

# safety: avoid division by zero
class_counts = np.where(class_counts == 0, 1, class_counts)

cw = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float)

# normalize weights
cw = cw / cw.sum() * NUM_LABELS
cw = cw.to(device)

print(f"Class weights → Non-Hate: {cw[0]:.3f} | Hate: {cw[1]:.3f}")

# =========================================
# LOSS FUNCTION (APPLY CLASS WEIGHTS)
# =========================================
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

# =========================================
# STORAGE DICTIONARIES
# =========================================
trained_models = {}     # {model_name: {best_path, tokenizer, best_f1}}
all_test_results = {}   # {model_name: {dataset: metrics}}

print("✅ Split + class weights ready.")

Train: 30,585 | Val: 3,399
Class weights → Non-Hate: 0.571 | Hate: 1.429
✅ Split + class weights ready.


In [11]:
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# 🔥 UPDATED TRAIN FUNCTION (ONLY CHANGE)
def run_train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    loader = tqdm(loader, desc="Training", leave=False)  # 👈 progress bar

    for batch in loader:
        batch.pop('raw_texts')
        labels = batch.pop('labels').to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        out  = model(**inputs, output_attentions=False, output_hidden_states=False)
        loss = loss_fn(out.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (out.logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)

        loader.set_postfix(loss=loss.item())  # 👈 live loss

    return total_loss / len(loader), correct / total


# =========================================
# MAIN TRAINING LOOP (UNCHANGED)
# =========================================
for model_tag, model_name in MODELS.items():
    print(f'\n{"="*65}')
    print(f'  TRAINING: {model_tag}')
    print(f'  HuggingFace: {model_name}')
    print(f'{"="*65}')

    model_dir = os.path.join(BASE_DIR, model_tag.replace('-', '_').replace(' ', '_'))
    best_ckpt = os.path.join(model_dir, 'best_checkpoint')
    os.makedirs(model_dir, exist_ok=True)

    # TOKENIZER
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # LOADERS
    tr_loader  = make_loader(X_tr,  y_tr,  tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=True)
    val_loader = make_loader(X_val, y_val, tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False)

    # MODEL
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS
    ).to(device)

    # OPTIMIZER + SCHEDULER
    no_decay = ['bias', 'LayerNorm.weight']

    opt_params = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]

    optimizer = AdamW(opt_params, lr=LEARNING_RATE)

    total_steps  = max(1, len(tr_loader) * EPOCHS)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

    # TRAIN LOOP
    history    = {'tr_loss':[], 'tr_acc':[], 'val_loss':[], 'val_f1':[], 'val_acc':[]}
    best_f1    = 0.0
    best_epoch = 0

    print(f'\n{"Ep":<5}{"TrLoss":<11}{"TrAcc":<11}{"VaLoss":<11}{"VaF1":<10}{"VaAcc"}')
    print('-' * 58)

    for ep in range(1, EPOCHS + 1):

        print(f"\n🔄 Running Epoch {ep}/{EPOCHS}...")  # 👈 added status

        tl, ta = run_train_epoch(
            model, tr_loader, optimizer, scheduler, loss_fn, device
        )

        vl, vf, va, _, _, _ = run_eval_epoch(
            model, val_loader, loss_fn, device
        )

        history['tr_loss'].append(tl)
        history['tr_acc'].append(ta)
        history['val_loss'].append(vl)
        history['val_f1'].append(vf)
        history['val_acc'].append(va)

        flag = ' ← best' if vf > best_f1 else ''
        print(f'{ep:<5}{tl:<11.4f}{ta:<11.4f}{vl:<11.4f}{vf:<10.4f}{va:.4f}{flag}')

        if vf > best_f1:
            best_f1 = vf
            best_epoch = ep

            model.save_pretrained(best_ckpt)
            tokenizer.save_pretrained(best_ckpt)

    print(f'\nBest: Epoch {best_epoch}  |  Val Macro F1 = {best_f1:.4f}')
    print(f'Checkpoint saved → {best_ckpt}/')

    trained_models[model_tag] = {
        'best_path':   best_ckpt,
        'tokenizer':   tokenizer,
        'history':     history,
        'best_val_f1': best_f1,
        'best_epoch':  best_epoch,
        'model_dir':   model_dir,
    }

    # CLEAN MEMORY
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('\n✅ Both models trained and saved.')


  TRAINING: HingRoBERTa
  HuggingFace: l3cube-pune/hing-roberta


tokenizer_config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

...


Ep   TrLoss     TrAcc      VaLoss     VaF1      VaAcc
----------------------------------------------------------

🔄 Running Epoch 1/4...


1    0.5250     0.7778     0.3346     0.8623    0.8856 ← best

🔄 Running Epoch 2/4...


2    0.3232     0.9019     0.3115     0.8958    0.9126 ← best

🔄 Running Epoch 3/4...


3    0.2689     0.9281     0.3342     0.9094    0.9253 ← best

🔄 Running Epoch 4/4...


4    0.2447     0.9383     0.3456     0.9067    0.9229

Best: Epoch 3  |  Val Macro F1 = 0.9094
Checkpoint saved → roberta_outputs/XLM_RoBERTa/best_checkpoint/

✅ Both models trained and saved.


In [12]:
# =========================================
# PLOT TRAINING CURVES (LOSS + ACCURACY)
# =========================================
eps = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -------- LOSS --------
axes[0].plot(eps, history['tr_loss'], marker='o', label='Train Loss')
axes[0].plot(eps, history['val_loss'], marker='o', label='Val Loss')
axes[0].set_title('Loss vs Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid()

# -------- ACCURACY --------
axes[1].plot(eps, history['tr_acc'], marker='o', label='Train Acc')
axes[1].plot(eps, history['val_acc'], marker='o', label='Val Acc')
axes[1].set_title('Accuracy vs Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid()

plt.suptitle(f'{model_tag} Training Performance', fontsize=14)
plt.tight_layout()

# SAVE GRAPH
graph_path = os.path.join(model_dir, 'training_graphs.png')
plt.savefig(graph_path, dpi=150)
plt.close()

print(f"📊 Training graphs saved → {graph_path}")

📊 Training graphs saved → roberta_outputs/XLM_RoBERTa/training_graphs.png


## Cell 9 — Cross-Lingual Testing
Evaluates each best checkpoint on English test set, then Hindi test set.
Saves per-sample predictions for XAI.

In [14]:
# =========================================
# CREATE TEST DATASETS
# =========================================
combined_df = pd.concat([
    pd.DataFrame({'text': X_val, 'label': y_val}),
    english_df,
    hindi_df
]).reset_index(drop=True)

TEST_SETS = {
    'hinglish_test': pd.DataFrame({'text': X_val, 'label': y_val}),
    'english': english_df,
    'hindi': hindi_df,
    'combined': combined_df
}

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# =========================================
# STORE RESULTS
# =========================================
test_results = {}

# =========================================
# TEST BOTH MODELS
# =========================================
for model_tag, info in trained_models.items():

    print(f'\n{"="*65}')
    print(f'🚀 TESTING MODEL: {model_tag}')
    print(f'{"="*65}')

    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path']
    ).to(device)

    tokenizer = info['tokenizer']
    model.eval()

    test_results[model_tag] = {}

    for split_name, df in TEST_SETS.items():

        print(f'\n→ {model_tag} on {split_name}')

        loader = make_loader(
            df['text'],
            df['label'],
            tokenizer,
            BATCH_SIZE,
            MAX_SEQ_LEN,
            shuffle=False
        )

        loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

        _, f1, acc, preds, labels, _ = run_eval_epoch(
            model, loader, loss_fn, device
        )

        precision = precision_score(labels, preds, average='macro', zero_division=0)
        recall    = recall_score(labels, preds, average='macro', zero_division=0)

        # PRINT METRICS
        print(f'Accuracy  : {acc:.4f}')
        print(f'Macro F1  : {f1:.4f}')
        print(f'Precision : {precision:.4f}')
        print(f'Recall    : {recall:.4f}')

        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=LABEL_NAMES, zero_division=0))

        # STORE RESULTS
        test_results[model_tag][split_name] = {
            'accuracy': acc,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

        # =========================================
        # CONFUSION MATRIX
        # =========================================
        cm = confusion_matrix(labels, preds)

        fig, ax = plt.subplots(figsize=(5, 5))
        ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(
            ax=ax, cmap='Blues', colorbar=False
        )

        ax.set_title(f'{model_tag} — {split_name}')
        plt.tight_layout()

        save_path = os.path.join(info['model_dir'], f'cm_{split_name}.png')
        plt.savefig(save_path, dpi=150)
        plt.close()

        print(f'Confusion matrix saved → {save_path}')

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Both models tested with full metrics.")


# =========================================
# TEST METRICS GRAPHS
# =========================================
import matplotlib.pyplot as plt

for dataset in TEST_SETS.keys():

    labels = []
    accs, f1s, precs, recs = [], [], [], []

    for model_tag in test_results:
        if dataset not in test_results[model_tag]:
            continue

        labels.append(model_tag)
        accs.append(test_results[model_tag][dataset]['accuracy'])
        f1s.append(test_results[model_tag][dataset]['f1'])
        precs.append(test_results[model_tag][dataset]['precision'])
        recs.append(test_results[model_tag][dataset]['recall'])

    x = range(len(labels))

    plt.figure(figsize=(8,5))

    plt.plot(x, accs, marker='o', label='Accuracy')
    plt.plot(x, f1s, marker='o', label='F1')
    plt.plot(x, precs, marker='o', label='Precision')
    plt.plot(x, recs, marker='o', label='Recall')

    plt.xticks(x, labels)
    plt.title(f'Model Comparison on {dataset}')
    plt.xlabel('Models')
    plt.ylabel('Score')
    plt.legend()
    plt.grid()

    save_path = os.path.join(BASE_DIR, f'test_metrics_{dataset}.png')
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"📊 Test graph saved → {save_path}")


🚀 TESTING MODEL: HingRoBERTa

→ HingRoBERTa on hinglish_test
Accuracy  : 0.9356
Macro F1  : 0.9218
Precision : 0.9180
Recall    : 0.9259

Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.96      0.95      0.95      2428
        Hate       0.88      0.90      0.89       971

    accuracy                           0.94      3399
   macro avg       0.92      0.93      0.92      3399
weighted avg       0.94      0.94      0.94      3399

Confusion matrix saved → roberta_outputs/HingRoBERTa/cm_hinglish_test.png

→ HingRoBERTa on english
Accuracy  : 0.4052
Macro F1  : 0.4019
Precision : 0.6048
Recall    : 0.6374

Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.22      0.99      0.36      4162
        Hate       0.99      0.29      0.45     20619

    accuracy                           0.41     24781
   macro avg       0.60      0.64      0.40     24781
weighted avg       0.86      0.41      

## Cell 10 — XAI Output Generation
For each model × each test set, saves a stratified sample of XAI-ready arrays:
- `cls_attn_row` — CLS token attention over all tokens → Attention heatmaps
- `token_ranking` — tokens ranked by CLS attention → AOPC input
- `cls_hidden` — CLS hidden state → SHAP feature space
- `last_layer_attn` — full last-layer attention matrix → attention viz
- `all_layers_attn` — all-layer averaged attention → attention rollout
- `logits`, `probs` — model confidence → AOPC probability drop

In [16]:
def generate_xai_outputs(model, tokenizer, texts, labels, model_tag,
                          split_name, model_dir, device,
                          n_samples=200):

    import os
    import numpy as np
    import pandas as pd
    import torch

    xai_dir = os.path.join(model_dir, f'xai_{split_name}')
    os.makedirs(xai_dir, exist_ok=True)

    # Stratified sampling
    df_tmp = pd.DataFrame({'text': texts, 'label': labels})
    n_per_class = n_samples // 2

    sample_df = (
        df_tmp.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=42))
        .reset_index(drop=True)
    )

    print(f'    XAI sample: {len(sample_df)} rows')

    model.eval()
    records = []

    for _, row in sample_df.iterrows():
        text = row['text']
        label = int(row['label'])

        enc = tokenizer(
            text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        enc_gpu = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(
                **enc_gpu,
                output_attentions=True,
                output_hidden_states=True
            )

        tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
        input_ids = enc['input_ids'][0].cpu().numpy()

        logits = out.logits[0].cpu().numpy()
        probs = torch.softmax(out.logits, dim=-1)[0].cpu().numpy()
        pred_label = int(np.argmax(logits))

        # Attention
        last_attn = out.attentions[-1][0].cpu().numpy()
        avg_attn = last_attn.mean(axis=0)
        cls_attn_row = avg_attn[0]

        # Hidden
        cls_hidden = out.hidden_states[-1][0, 0, :].cpu().numpy()

        records.append({
            'text': text,
            'true_label': label,
            'pred_label': pred_label,
            'tokens': tokens,
            'input_ids': input_ids,
            'logits': logits,
            'probs': probs,
            'cls_attn_row': cls_attn_row,
            'cls_hidden': cls_hidden
        })

    # Save main file
    np.savez(
        os.path.join(xai_dir, 'xai_samples.npz'),
        texts=np.array([r['text'] for r in records], dtype=object),
        true_labels=np.array([r['true_label'] for r in records]),
        pred_labels=np.array([r['pred_label'] for r in records]),
        logits=np.array([r['logits'] for r in records]),
        probs=np.array([r['probs'] for r in records]),
        cls_attn_rows=np.array([r['cls_attn_row'] for r in records]),
        cls_hiddens=np.array([r['cls_hidden'] for r in records]),
        allow_pickle=True
    )

    print(f'    Saved → {xai_dir}/')
    return records

In [17]:
# =========================================
# XAI CONFIG
# =========================================
RUN_XAI_ON = ['train_sample', 'hinglish_test', 'english', 'hindi']

# =========================================
# CREATE DATASETS FOR XAI
# =========================================
train_sample_df = pd.DataFrame({
    'text': X_tr,
    'label': y_tr
}).sample(n=min(XAI_SAMPLE_SIZE, len(X_tr)), random_state=RANDOM_SEED)

hinglish_test_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

combined_df = pd.concat([
    hinglish_test_df,
    english_df,
    hindi_df
]).reset_index(drop=True)

XAI_DATASETS = {
    'train_sample': train_sample_df,
    'hinglish_test': hinglish_test_df,
    'english': english_df,
    'hindi': hindi_df,
    'combined': combined_df
}

# =========================================
# RUN XAI FOR BOTH MODELS
# =========================================
for model_tag, info in trained_models.items():

    print(f'\n{"="*65}')
    print(f'🔍 RUNNING XAI FOR: {model_tag}')
    print(f'{"="*65}')

    tokenizer = info['tokenizer']
    model_dir = info['model_dir']

    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path'],
        output_attentions=True,
        output_hidden_states=True
    ).to(device)

    model.eval()

    for split_name, df in XAI_DATASETS.items():

        if split_name not in RUN_XAI_ON:
            continue

        print(f'\n→ XAI on {split_name} ({len(df):,} samples)')

        generate_xai_outputs(
            model=model,
            tokenizer=tokenizer,
            texts=df['text'].tolist(),
            labels=df['label'].tolist(),
            model_tag=model_tag,
            split_name=split_name,
            model_dir=model_dir,
            device=device,
            n_samples=XAI_SAMPLE_SIZE
        )

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ XAI generation complete for both models (including Hindi).")


🔍 RUNNING XAI FOR: HingRoBERTa

→ XAI on train_sample (200 samples)
    XAI sample: 160 rows
    Saved → roberta_outputs/HingRoBERTa/xai_train_sample/

→ XAI on hinglish_test (3,399 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_hinglish_test/

→ XAI on english (24,781 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_english/

→ XAI on hindi (8,192 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_hindi/

🔍 RUNNING XAI FOR: XLM-RoBERTa

→ XAI on train_sample (200 samples)
    XAI sample: 160 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_train_sample/

→ XAI on hinglish_test (3,399 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_hinglish_test/

→ XAI on english (24,781 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_english/

→ XAI on hindi (8,192 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_hindi/

✅ XAI

## Cell 11 — AOPC (Attention-Based)
Computes the Area Over the Perturbation Curve using the pre-saved token rankings.
When SHAP rankings are available, pass them via `shap_rankings` to get SHAP-AOPC for comparison.

In [18]:
def compute_aopc(model, tokenizer, records, device,
                 steps=AOPC_STEPS, shap_rankings=None, label='attention'):
    """
    AOPC: mask top-k tokens one-by-one, measure confidence drop.

    Args:
        records       : list of dicts from generate_xai_outputs()
        shap_rankings : optional List[List[int]] of SHAP-based token rankings
                        (replaces attention rankings if provided)
        label         : string tag for this run ('attention' or 'shap')
    Returns:
        dict: mean_aopc, all_scores (per sample), all_curves (per step)
    """
    model.eval()
    mask_token = tokenizer.mask_token or tokenizer.unk_token
    all_scores, all_curves = [], []

    for i, rec in enumerate(records):
        text         = rec['text']
        pred_class   = rec['pred_label']
        orig_prob    = float(rec['probs'][pred_class])

        # Choose ranking source
        if shap_rankings is not None:
            ranking = list(shap_rankings[i])
        else:
            ranking = list(rec['token_ranking'])

        words        = text.split()
        masked_words = words.copy()
        curve        = []

        def subtoken_to_word_idx(subtoken_idx):
            """Map subword token index → word index (approximate)."""
            real = int(subtoken_idx) - 1   # -1 for [CLS]
            if real < 0: return None
            cumulative = 0
            for w_idx, word in enumerate(words):
                cumulative += len(tokenizer.tokenize(word))
                if real < cumulative:
                    return w_idx
            return None

        for k in range(1, steps + 1):
            if k <= len(ranking):
                w_idx = subtoken_to_word_idx(ranking[k - 1])
                if w_idx is not None and w_idx < len(masked_words):
                    masked_words[w_idx] = mask_token

            perturbed = ' '.join(masked_words)
            enc = tokenizer(perturbed, return_tensors='pt',
                            max_length=MAX_SEQ_LEN, truncation=True,
                            padding='max_length')
            enc = {k2: v.to(device) for k2, v in enc.items()}
            with torch.no_grad():
                out = model(**enc, output_attentions=False, output_hidden_states=False)
            p = float(torch.softmax(out.logits, -1)[0][pred_class].cpu())
            curve.append(orig_prob - p)

        all_scores.append(float(np.mean(curve)))
        all_curves.append(curve)

    return {
        'label':      label,
        'mean_aopc':  float(np.mean(all_scores)),
        'std_aopc':   float(np.std(all_scores)),
        'all_scores': all_scores,
        'all_curves': all_curves,
    }

print('AOPC function ready.')

AOPC function ready.


In [27]:
# =========================================
# AOPC RESULTS STORAGE
# =========================================
aopc_results_all = {}

for model_tag, info in trained_models.items():

    aopc_results_all[model_tag] = {}

    model_dir = info['model_dir']
    tokenizer = info['tokenizer']

    print(f'\n{"="*65}')
    print(f'  AOPC: {model_tag}')
    print(f'{"="*65}')

    # 🔥 Load model (LIGHT version)
    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path'],
        output_attentions=False,
        output_hidden_states=False
    ).to(device)

    model.eval()

    for split_name in ['english', 'hindi']:   # 🔥 limit for safety

        xai_dir = os.path.join(model_dir, f'xai_{split_name}')
        xai_file = os.path.join(xai_dir, 'xai_samples.npz')

        # ❗ Skip if XAI not present
        if not os.path.exists(xai_file):
            print(f"⚠️ Skipping {split_name} (no XAI file found)")
            continue

        print(f'\n  {split_name} — attention-based AOPC')

        # =========================================
        # LOAD XAI DATA (FROM DISK, NOT RAM)
        # =========================================
        data = np.load(xai_file, allow_pickle=True)

        records = []
        for i in range(len(data['texts'])):

            attn = data['cls_attn_rows'][i]

            # 🔥 create ranking (highest attention first)
            token_ranking = list(np.argsort(-attn))

            records.append({
                'text': data['texts'][i],
                'true_label': int(data['true_labels'][i]),
                'pred_label': int(data['pred_labels'][i]),
                'cls_attn_row': attn,
                'probs': data['probs'][i],
                'token_ranking': token_ranking
            })

        # =========================================
        # COMPUTE AOPC
        # =========================================
        attn_result = compute_aopc(
            model,
            tokenizer,
            records,
            device,
            label='attention'
        )

        print(f'    Mean AOPC: {attn_result["mean_aopc"]:.4f} ± {attn_result["std_aopc"]:.4f}')

        aopc_results_all[model_tag][split_name] = {
            'attention': attn_result
        }

        # =========================================
        # PLOT AOPC CURVE
        # =========================================
        mean_curve = np.mean(attn_result['all_curves'], axis=0)

        plt.figure(figsize=(7, 4))
        plt.plot(range(1, AOPC_STEPS + 1), mean_curve,
                 marker='o', color='darkorange')

        plt.xlabel('Tokens masked (k)')
        plt.ylabel('Mean probability drop')
        plt.title(f'AOPC — {model_tag} | {split_name}')
        plt.grid(alpha=0.3)
        plt.tight_layout()

        plt.savefig(os.path.join(xai_dir, 'aopc_attention.png'), dpi=150)
        plt.close()

        # =========================================
        # SAVE AOPC RESULTS
        # =========================================
        np.savez(
            os.path.join(xai_dir, 'aopc_attention.npz'),
            mean_aopc  = np.array(attn_result['mean_aopc']),
            std_aopc   = np.array(attn_result['std_aopc']),
            all_scores = np.array(attn_result['all_scores']),
            all_curves = np.array(attn_result['all_curves']),
        )

        print(f"    Saved → {xai_dir}/aopc_attention.*")

    # ✅ CORRECT POSITION (outside inner loop)
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('\n✅ AOPC (attention) complete.')


  AOPC: HingRoBERTa

  english — attention-based AOPC
    Mean AOPC: 0.1547 ± 0.3045
    Saved → roberta_outputs/HingRoBERTa/xai_english/aopc_attention.*

  hindi — attention-based AOPC
    Mean AOPC: 0.1303 ± 0.1934
    Saved → roberta_outputs/HingRoBERTa/xai_hindi/aopc_attention.*

  AOPC: XLM-RoBERTa

  english — attention-based AOPC
    Mean AOPC: 0.0910 ± 0.2897
    Saved → roberta_outputs/XLM_RoBERTa/xai_english/aopc_attention.*

  hindi — attention-based AOPC
    Mean AOPC: 0.0041 ± 0.0907
    Saved → roberta_outputs/XLM_RoBERTa/xai_hindi/aopc_attention.*

✅ AOPC (attention) complete.


## Cell 12 — Results Summary Table

In [29]:
rows = []

for model_tag, info in trained_models.items():
    for split_name in TEST_SETS:

        # 🔥 skip if test results not present
        if split_name not in test_results[model_tag]:
            continue

        test_res = test_results[model_tag][split_name]

        # 🔥 safe AOPC access
        if (model_tag in aopc_results_all and
            split_name in aopc_results_all[model_tag]):

            aopc_res = aopc_results_all[model_tag][split_name]['attention']
            aopc_val = round(aopc_res['mean_aopc'], 4)
        else:
            aopc_val = None   # or 'N/A'

        rows.append({
            'model':          model_tag,
            'train_data':     'Hinglish (no norm)',
            'test_data':      split_name,
            'normalization':  False,
            'best_val_f1':    round(info['best_val_f1'], 4),
            'best_epoch':     info['best_epoch'],
            'test_accuracy':  round(test_res['accuracy'], 4),
            'test_macro_f1':  round(test_res['f1'], 4),   # 🔥 fixed
            'test_precision': round(test_res['precision'], 4),
            'test_recall':    round(test_res['recall'], 4),
            'aopc_attention': aopc_val,
            'aopc_shap':      'TBD',
        })

# CREATE DF
summary_df = pd.DataFrame(rows)

# SAVE
summary_csv = os.path.join(BASE_DIR, 'full_results.csv')
summary_df.to_csv(summary_csv, index=False)

print('=== FULL RESULTS SUMMARY ===')
print(summary_df.to_string(index=False))

=== FULL RESULTS SUMMARY ===
      model         train_data     test_data  normalization  best_val_f1  best_epoch  test_accuracy  test_macro_f1  test_precision  test_recall  aopc_attention aopc_shap
HingRoBERTa Hinglish (no norm) hinglish_test          False       0.9218           4         0.9356         0.9218          0.9180       0.9259             NaN       TBD
HingRoBERTa Hinglish (no norm)       english          False       0.9218           4         0.4052         0.4019          0.6048       0.6374          0.1547       TBD
HingRoBERTa Hinglish (no norm)         hindi          False       0.9218           4         0.5194         0.4226          0.4890       0.4952          0.1303       TBD
HingRoBERTa Hinglish (no norm)      combined          False       0.9218           4         0.4805         0.4770          0.6342       0.6093             NaN       TBD
XLM-RoBERTa Hinglish (no norm) hinglish_test          False       0.9094           3         0.9253         0.9094       

## Cell 13 — Download Everything

In [ ]:
import shutil
from google.colab import files

zip_name = 'roberta_outputs_backup'   # better name (avoid overwrite confusion)

print("📦 Creating zip...")

shutil.make_archive(zip_name, 'zip', BASE_DIR)

print(f"✅ Zipped: {zip_name}.zip")

files.download(f'{zip_name}.zip')

print("⬇️ Download triggered.")

📦 Creating zip...


---
## XAI Module Reference Card

### Directory layout after running this notebook
```
roberta_outputs/
├── full_results.csv
├── HingRoBERTa/
│   ├── best_checkpoint/          ← reload with AutoModelForSequenceClassification
│   ├── training_curves.png
│   ├── cm_english.png
│   ├── cm_hindi.png
│   ├── predictions_english.csv
│   ├── predictions_hindi.csv
│   ├── xai_english/
│   │   ├── xai_samples.npz       ← main XAI arrays
│   │   ├── tokens_list.npy
│   │   ├── token_rankings.npy    ← AOPC input (attention-ranked)
│   │   ├── last_layer_attn.npy
│   │   ├── all_layers_attn.npy
│   │   ├── aopc_attention.npz
│   │   └── aopc_attention.png
│   └── xai_hindi/  (same structure)
└── XLM_RoBERTa/ (same structure)
```

### Loading in `xai/explain.py`
```python
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_PATH = 'roberta_outputs/HingRoBERTa/best_checkpoint'
XAI_PATH   = 'roberta_outputs/HingRoBERTa/xai_english'

model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, output_attentions=True, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Fixed-shape arrays
xai           = np.load(f'{XAI_PATH}/xai_samples.npz', allow_pickle=True)
texts         = xai['texts']           # (N,) raw strings
true_labels   = xai['true_labels']     # (N,)
pred_labels   = xai['pred_labels']     # (N,)
probs         = xai['probs']           # (N, 2)
cls_attn_rows = xai['cls_attn_rows']   # (N, seq_len) → attention bars
cls_hiddens   = xai['cls_hiddens']     # (N, 768)     → SHAP feature space

# Variable-length arrays
tokens_list     = np.load(f'{XAI_PATH}/tokens_list.npy',    allow_pickle=True)  # (N,) lists
token_rankings  = np.load(f'{XAI_PATH}/token_rankings.npy', allow_pickle=True)  # (N,) ranked idx
last_attn       = np.load(f'{XAI_PATH}/last_layer_attn.npy',allow_pickle=True)  # (N,) (heads,seq,seq)
all_attn        = np.load(f'{XAI_PATH}/all_layers_attn.npy',allow_pickle=True)  # (N,) (layers,seq,seq)
```

### SHAP text explainer
```python
import shap, torch

def predict_fn(text_list):
    enc = tokenizer(list(text_list), return_tensors='pt',
                    padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**enc)
    return torch.softmax(out.logits, -1).numpy()

explainer   = shap.Explainer(predict_fn, shap.maskers.Text(tokenizer))
shap_values = explainer(texts[:50])

# Extract SHAP token rankings for SHAP-AOPC
shap_rankings = [
    np.argsort(np.abs(shap_values[i, :, 1].values))[::-1].tolist()
    for i in range(len(texts[:50]))
]
# Then pass to compute_aopc(..., shap_rankings=shap_rankings, label='shap')
```

### Key flag reminder
Both models are saved with `output_attentions=True` and `output_hidden_states=True` in their config.
These flags are also passed explicitly at inference time in this notebook.
Your XAI teammate does **not** need to change anything — just reload and call `model(**enc, output_attentions=True)`.